# Transcript Matching — Final Model Training
Trains the production logistic-regression classifier on all available labeled pairs  
(YouTube-only + YouTube×TikTok), picks the best probability threshold, and saves the artefacts.

**Inputs:** labeled annotation files, all transcripts  
**Outputs:** `data/model.pkl`, `data/best_thr.txt`, diagnostic plots  

## 1. Imports & Setup

In [1]:
import os
import sys
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, confusion_matrix

module_path = os.path.abspath(os.path.join(''))
if module_path not in sys.path:
    sys.path.append(module_path)

if os.path.isdir("../data/"):
    os.chdir("../")

from src.utils import *
from src.plots_utils import *
from src.transcript_matching import *
from src.file_io import *

%load_ext autoreload
%autoreload 2

rcParams, column_width, page_width, shorts_style, RV_style = get_plt_attr()
plt.rcParams.update(rcParams)

logger = logging.getLogger(__name__)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename='logs/transcript_matching.log',
    level=logging.DEBUG,
    format='%(asctime)s:%(levelname)s:%(message)s',
    datefmt='%m-%d-%Y %I:%M:%S'
)

## 2. Load Data

In [2]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_tt_channels = pd.read_json('data/tiktok/channels/news_channels.json')
pp_tt_channels = pd.read_json('data/tiktok/channels/pp_channels.json')

In [3]:
# --- transcripts ---
yt_nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_videos_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl", lines=True)
yt_pp_videos_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl", lines=True)

tt_nm_videos_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_videos_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_videos_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl", lines=True)
tt_pp_videos_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl", lines=True)
for tt_df in [tt_nm_videos_2022, tt_nm_videos_2024, tt_pp_videos_2022, tt_pp_videos_2024]:
    tt_df.rename(columns={'id':'videoId', 'voice_to_text':'transcript','create_time':'publishedAt'}, inplace=True)

all_transcripts = pd.concat([
    yt_nm_videos_2022[['videoId', 'transcript']], yt_pp_videos_2022[['videoId', 'transcript']],
    tt_nm_videos_2022[['videoId', 'transcript']], tt_pp_videos_2022[['videoId', 'transcript']],
    yt_nm_videos_2024[['videoId', 'transcript']], yt_pp_videos_2024[['videoId', 'transcript']],
    tt_nm_videos_2024[['videoId', 'transcript']], tt_pp_videos_2024[['videoId', 'transcript']],
]).dropna(subset='transcript')


all_transcripts['transcript_clean'] = all_transcripts['transcript'].apply(
    clean_transcripts,
    strip_accents=True, lowercase=True, remove_brackets=True, remove_openers=False
)
print(f"{len(all_transcripts):,} transcripts loaded")

91,701 transcripts loaded


## 3. Build Training Set

In [4]:
def prepare_labeled_pairs(labeled_pairs: pd.DataFrame, transcripts: pd.DataFrame) -> pd.DataFrame:
    """Merge transcript text into a labeled-pairs dataframe and add binary label."""
    initial_length = len(labeled_pairs)
    transcripts['videoId'] = transcripts['videoId'].apply(str)
    labeled_pairs.dropna(subset="Label", inplace=True)

    valid_labels = {"No", "Yes, one whole fragment", "Yes, cut fragments", "Yes"}
    assert set(labeled_pairs.Label) - valid_labels == set(), set(labeled_pairs.Label)

    id_name = 'id' if ('id1' in labeled_pairs.columns) else 'videoId'

    def clean_id(x):
        return (x.replace('www.youtube.com/shorts/', '')
                 .replace('www.youtube.com/watch?v=', '')
                 .replace('www.tiktok.com/@user/video/', ''))

    labeled_pairs['videoId1'] = labeled_pairs[f'{id_name}1'].apply(clean_id)
    labeled_pairs['videoId2'] = labeled_pairs[f'{id_name}2'].apply(clean_id)
    labeled_pairs['id1,id2'] = labeled_pairs['videoId1'] + labeled_pairs['videoId2']

    labeled_pairs = (
        labeled_pairs
        .merge(transcripts[['videoId', 'transcript', 'transcript_clean']],
               left_on='videoId1', right_on='videoId')
        .rename(columns={'transcript_clean': 'transcript_clean1', 'transcript': 'transcript1'})
        .drop(columns=['videoId'])
    )
    labeled_pairs = (
        labeled_pairs
        .merge(transcripts[['videoId', 'transcript', 'transcript_clean']],
               left_on='videoId2', right_on='videoId')
        .rename(columns={'transcript_clean': 'transcript_clean2', 'transcript': 'transcript2'})
        .drop(columns=['videoId'])
    )

    if len(labeled_pairs) != initial_length:
        print(f"Length mismatch: {initial_length} → {len(labeled_pairs)}")

    labeled_pairs = labeled_pairs[labeled_pairs.transcript1.str.len() > 50]
    labeled_pairs = labeled_pairs[labeled_pairs.transcript2.str.len() > 50]
    labeled_pairs['label_binary'] = (labeled_pairs['Label'] != 'No').astype(int)
    return labeled_pairs

In [5]:
same_channels_nm = prepare_labeled_pairs(
    pd.read_excel("data/youtube/transcripts/manual_annotations/similarities_same_channel_news_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

same_channels_pp = prepare_labeled_pairs(
    pd.read_excel( "data/youtube/transcripts/manual_annotations/similarities_same_channel_polit_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

diff_channels_yt = prepare_labeled_pairs(
    pd.read_excel("data/youtube/transcripts/manual_annotations/similarities_diff_channel_polit_2024_labeled.xlsx", index_col=0), 
    all_transcripts
)

Length mismatch: 65 → 64


In [6]:

labeled_pairs_youtube_only = pd.concat([same_channels_nm, same_channels_pp, diff_channels_yt
    
]).reset_index(drop=True)

labeled_pairs_youtube_only['transcript_cut1'] = labeled_pairs_youtube_only['transcript_clean1'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_only['transcript_cut2'] = labeled_pairs_youtube_only['transcript_clean2'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t1', 'transcript_cut1', 'transcript_clean2')
labeled_pairs_youtube_only = compute_ratios(labeled_pairs_youtube_only, '_cut_t2', 'transcript_clean1', 'transcript_cut2')



In [8]:
# YouTube × TikTok pairs
tt_annotation_files = [
    "data/pairs_of_transcripts/annotation/pairs_yt_tt_nm_2022_to_annotate.xlsx",
    "data/pairs_of_transcripts/annotation/pairs_yt_tt_pp_2024_to_annotate.xlsx",
]
labeled_pairs_youtube_tiktok = pd.concat([
    prepare_labeled_pairs(pd.read_excel(f, index_col=0), all_transcripts)
    for f in tt_annotation_files
]).reset_index(drop=True)

labeled_pairs_youtube_tiktok['transcript_cut1'] = labeled_pairs_youtube_tiktok['transcript_clean1'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_tiktok['transcript_cut2'] = labeled_pairs_youtube_tiktok['transcript_clean2'].apply(
    extract_short_transcript, length_transcript=2000)
labeled_pairs_youtube_tiktok = compute_ratios(labeled_pairs_youtube_tiktok, '_cut_t1', 'transcript_cut1', 'transcript_clean2')
labeled_pairs_youtube_tiktok = compute_ratios(labeled_pairs_youtube_tiktok, '_cut_t2', 'transcript_clean1', 'transcript_cut2')

print(f"YouTube × TikTok pairs: {len(labeled_pairs_youtube_tiktok):,}")

YouTube × TikTok pairs: 80


In [9]:
print(f"YouTube-only NM pairs {len(same_channels_nm)}")
print(f"YouTube-only PP pairs {len(same_channels_pp)}")
print(f"YouTube-only NM x PP pairs {len(diff_channels_yt)}")
print(f"YouTube × TikTok pairs: {len(labeled_pairs_youtube_tiktok)}")
print(f"All pairs: {len(same_channels_nm) + len(same_channels_pp) + len(diff_channels_yt) + len(labeled_pairs_youtube_tiktok)}")

YouTube-only NM pairs 87
YouTube-only PP pairs 60
YouTube-only NM x PP pairs 81
YouTube × TikTok pairs: 80
All pairs: 308


In [18]:
# Merge + strip fw. prefix for consistency
for df in [labeled_pairs_youtube_only, labeled_pairs_youtube_tiktok]:
    df.columns = [c.replace('fw.', '') for c in df.columns]

common_columns = list(set(labeled_pairs_youtube_only.columns) & set(labeled_pairs_youtube_tiktok.columns))
all_labeled_pairs = pd.concat([labeled_pairs_youtube_only, labeled_pairs_youtube_tiktok])
print(f"Total labeled pairs: {len(all_labeled_pairs)}, {len(all_labeled_pairs[all_labeled_pairs["label_binary"] == 1])} positive and " + 
      f"{len(all_labeled_pairs[all_labeled_pairs["label_binary"] == 0])} negative")


Total labeled pairs: 308, 111 positive and 197 negative


## 4. Add Chunk-Match Features

In [ ]:
all_labeled_pairs['chunk_match_score_1'] = all_labeled_pairs.apply(
    lambda row: best_chunk_match(row['transcript_cut1'], row['transcript_clean2']), axis=1)
all_labeled_pairs['chunk_match_score_2'] = all_labeled_pairs.apply(
    lambda row: best_chunk_match(row['transcript_clean1'], row['transcript_cut2']), axis=1)

In [ ]:
labeled_pairs = process_pairs(all_labeled_pairs, video2channel_yt)
labeled_pairs.drop(columns=['party', 'id1', 'id2', 'partialratio', 'Unnamed: 6'],
                   inplace=True, errors='ignore')

print(f"Class balance:")
display(labeled_pairs.groupby(['diffchannel', 'label_binary']).size().to_frame('count'))
print(f"Total: {len(labeled_pairs):,}")

## 5. Train Final Model

In [ ]:
# ── Feature selection ──────────────────────────────────────────────────────
# Adjust based on results from 01_matching_methods_evaluation.ipynb
FEATURE_COLUMNS = [
    'partialratio_cut_t1',
    'partialratio_cut_t2',
    # 'sortratio_cut_t1',
    # 'sortratio_cut_t2',
    # 'setratio_cut_t1',
    # 'setratio_cut_t2',
    'chunk_match_score_1',
    'chunk_match_score_2',
    # 'diffchannel',
]

X_all = labeled_pairs[FEATURE_COLUMNS]
y_all = labeled_pairs['label_binary']

model = LogisticRegression()
model.fit(X_all, y_all)

print("Model coefficients:")
for feat, coef in zip(model.feature_names_in_, model.coef_[0]):
    print(f"  {feat:30s} {coef:+.4f}")
print(f"  {'intercept':30s} {model.intercept_[0]:+.4f}")

## 6. Threshold Selection

In [ ]:
predicted_probas = model.predict_proba(X_all)[:, 1]
prec, rec, thr = precision_recall_curve(y_all, predicted_probas)
f1 = 2 * prec * rec / (prec + rec)
best_thr = np.mean([thr[np.argmax(f1)], thr[np.argmax(f1) - 1]])

print(f"Best threshold: {best_thr:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(thr, f1[1:], '.')
plt.axvline(best_thr, color='k', linestyle='--', label=f'best thr = {best_thr:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1')
plt.title('F1 vs threshold (train set)')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Diagnostics

In [ ]:
labeled_pairs['predicted_proba'] = predicted_probas
labeled_pairs['predicted_label'] = (predicted_probas >= best_thr).astype(int)

acc = (labeled_pairs['predicted_label'] == y_all).mean()
print(f"Train accuracy at best_thr: {acc:.4f}")
print("\nConfusion matrix (train):")
print(confusion_matrix(y_all, labeled_pairs['predicted_label']))

In [ ]:
# Decision-boundary histogram
fig, ax = plt.subplots(figsize=(column_width, 1.5))
bin_edges = np.linspace(0, 1, 50)

ax.axvspan(-0.01, best_thr, color='tab:blue', alpha=0.1)
ax.axvspan(best_thr, 1.01, color='tab:red',  alpha=0.1)

ax.hist(labeled_pairs[y_all == 0]['predicted_proba'], bins=bin_edges, label='non-matching pairs')
ax.hist(labeled_pairs[y_all == 1]['predicted_proba'], bins=bin_edges, label='matching pairs')
ax.axvline(best_thr, color='k', linestyle='--')

ax.set_xlabel('Predicted combined ratio')
ax.set_ylabel('Count')
ax.set_yscale('log')
ax.set_xlim(-0.01, 1.01)
ax.set_xticks(
    [0, 0.2, 0.4, 0.6, 0.8, 1, best_thr],
    ['0', '0.2', '0.4', '0.6', '0.8', '1', r'$\tau$' + f'\n(={best_thr:.2f})']
)
ax.text(best_thr / 2, 65, 'NEGATIVE', color='tab:blue', fontdict={'fontweight': 'bold'}, ha='center')
ax.text(best_thr + (1 - best_thr) / 2, 65, 'POSITIVE', color='tab:orange', fontdict={'fontweight': 'bold'}, ha='center')
ax.legend(loc=(0.2, 0.4))
plt.tight_layout(pad=0, w_pad=-0.5, h_pad=0)
fig.savefig('Figures/decision_boundary_plot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Inspect misclassified pairs
misclassified_cols = [
    'label_binary', 'videoId1', 'videoId2',
    'transcript_clean1', 'transcript_clean2',
    'partialratio_cut_t1', 'partialratio_cut_t2',
    'chunk_match_score_1', 'chunk_match_score_2',
    'predicted_proba', 'predicted_label',
]
misclassified = labeled_pairs[
    labeled_pairs['predicted_label'] != labeled_pairs['label_binary']
][misclassified_cols]
print(f"{len(misclassified)} misclassified pairs")
display(misclassified)

## 8. Save Model Artefacts

In [ ]:
with open('data/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('data/best_thr.txt', 'w') as f:
    f.write(str(best_thr))

print(f"Saved model → data/model.pkl")
print(f"Saved threshold ({best_thr:.4f}) → data/best_thr.txt")